# v1 Kaggle Spike — Clinical Transcript -> SOAP JSON

**Goal:** prove the SFT pipeline end-to-end on Kaggle T4 AND test true generalisation via held-out transcript shapes.

**What this proves**
- Repo + secret + HF install works on Kaggle
- Qwen 2.5 3B Instruct + 4-bit (bitsandbytes) + LoRA (peft) trains on toy clinical data via TRL SFTTrainer
- Adapter saves to `/kaggle/working/adapter_v0`
- Eval loop reports 4 metrics on held-out shapes — measures true generalisation, not memorisation

**Held-out eval design**
- 15 transcript shapes total. 2 are held out by `random.Random(42).sample(range(15), 2)` -> `[1, 10]` (shape02 + shape11).
- Train: 500 examples drawn from the OTHER 13 shapes.
- Eval: 50 examples drawn ONLY from the 2 held-out shapes — model has never seen this format.

**What this does NOT prove**
- Template-awareness — only one output spec (SOAP) in v1; deferred to v2
- Generalisation to real transcripts — toy transcripts are short and synthetic
- Production quality — n=50 eval, 150 train steps

**Success criteria** (rough, on held-out shapes): json_parse >= 0.8, schema_keys >= 0.7, evidence_grounding >= 0.5, cc_overlap >= 0.4. Anything substantially above these means real generalisation; below means template overfit (still publishable as honest signal).

In [ ]:
# Cell 2 — env check
import sys
print('Python:', sys.version)
!nvidia-smi

In [ ]:
# Cell 3 — clone repo via GITHUB_PAT secret
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT = UserSecretsClient().get_secret('GITHUB_PAT')
REPO = 'nizamphoenix/clinical_transcript_summariser'
DEST = '/kaggle/working/repo'

if os.path.exists(DEST):
    subprocess.run(['rm', '-rf', DEST], check=True)

url = f'https://{PAT}@github.com/{REPO}.git'
subprocess.run(['git', 'clone', '--depth', '1', url, DEST], check=True)
print('cloned ->', DEST)
%cd $DEST

In [ ]:
# Cell 4 — install local package + missing HF deps (Kaggle image lacks bnb/trl current)
!pip install -q -U "bitsandbytes>=0.46.1" "trl>=1.0"
!pip install -q -e .

In [ ]:
# Cell 5 — generate toy data with HELD-OUT shapes for true generalisation eval.
# Train uses 13 shapes; eval uses ONLY the 2 held-out shapes (chosen by
# random.Random(42).sample(range(15), 2) -> indices [1, 10] = shape02 + shape11).
!python scripts/gen_toy.py --n-train 500 --n-eval 50 \
    --held-out 2 --holdout-seed 42 --seed 0 --out-dir data/
!ls -la data/

In [ ]:
# Cell 6 — load Qwen 2.5 3B Instruct, 4-bit (HF native)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MAX_SEQ_LEN = 2048
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
print('loaded', MODEL_NAME)

In [ ]:
# Cell 7 — attach LoRA adapters (peft)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 8 — format toy data with chat template
import json
from datasets import Dataset
from src.prompts import build_messages

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

train_rows = load_jsonl('data/toy.train.jsonl')
eval_rows  = load_jsonl('data/toy.eval.jsonl')

def to_text(row):
    msgs = build_messages(row['transcript'])
    msgs.append({'role':'assistant',
                 'content': json.dumps(row['soap'], ensure_ascii=False)})
    return tokenizer.apply_chat_template(msgs, tokenize=False)

train_ds = Dataset.from_list([{'text': to_text(r)} for r in train_rows])
eval_ds  = Dataset.from_list([{'text': to_text(r)} for r in eval_rows])
print('n_train =', len(train_ds), ' n_eval =', len(eval_ds))
print('---example 0 (first 600 chars)---')
print(train_ds[0]['text'][:600])

In [ ]:
# Cell 9 — SFTTrainer config (TRL 1.x API: SFTConfig + processing_class)
from trl import SFTTrainer, SFTConfig

# === flip MAX_STEPS=4 (smoke) -> 150 (real run) ===
MAX_STEPS  = 4
EVAL_EVERY = max(1, MAX_STEPS // 6)   # ~6 eval points across the run
print(f'MAX_STEPS={MAX_STEPS}  EVAL_EVERY={EVAL_EVERY}')

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        output_dir='/kaggle/working/sft_out',
        dataset_text_field='text',
        max_length=MAX_SEQ_LEN,
        packing=False,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        eval_strategy='steps',
        eval_steps=EVAL_EVERY,
        save_strategy='steps',
        save_steps=EVAL_EVERY,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=0,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        report_to='none',
    ),
)

In [ ]:
# Cell 10 — train
trainer.train()

## Eval loop

Four metrics on 10 held-out examples:
1. **json_parse_rate** — output is valid JSON
2. **schema_keys_rate** — top-level has subjective/objective/assessment/plan
3. **cc_overlap_mean** — Jaccard on chief_complaint.text tokens
4. **evidence_grounding_rate** — fraction of leaf evidence_spans that are substrings of the transcript (hallucination guardrail)

In [ ]:
# Cell 12 — eval loop
from src.toy_data import _collect_spans
from src.prompts import build_messages

model.eval()

def generate(transcript: str) -> str:
    msgs = build_messages(transcript)
    inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    ).to('cuda')
    out = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def score(pred_str: str, gold: dict, transcript: str) -> dict:
    # (1) JSON parse
    try:
        pred = json.loads(pred_str)
    except Exception:
        return {'parse': 0, 'keys': 0, 'cc': 0.0, 'ground': 0.0}
    # (2) schema keys
    keys = int(set(pred.keys()) >= {'subjective','objective','assessment','plan'})
    # (3) CC token overlap (Jaccard)
    try:
        p_cc = (pred['subjective']['chief_complaint']['text'] or '').lower()
        g_cc = (gold['subjective']['chief_complaint']['text'] or '').lower()
        a, b = set(p_cc.split()), set(g_cc.split())
        cc = len(a & b) / max(len(a | b), 1)
    except Exception:
        cc = 0.0
    # (4) evidence grounding
    spans = [s for s in _collect_spans(pred) if s]
    if not spans:
        ground = 0.0
    else:
        ground = sum(1 for s in spans if s in transcript) / len(spans)
    return {'parse': 1, 'keys': keys, 'cc': cc, 'ground': ground}

results = []
preds = []
# === SMOKE: subsample to 10 rows. Restore to `eval_rows` for the 150-step real run. ===
EVAL_SUBSET = eval_rows[:10]
for row in EVAL_SUBSET:
    p = generate(row['transcript'])
    preds.append(p)
    results.append(score(p, row['soap'], row['transcript']))

n = len(results)
summary = {
    'n': n,
    'json_parse_rate':         sum(r['parse']  for r in results) / n,
    'schema_keys_rate':        sum(r['keys']   for r in results) / n,
    'cc_overlap_mean':         sum(r['cc']     for r in results) / n,
    'evidence_grounding_rate': sum(r['ground'] for r in results) / n,
}
print(json.dumps(summary, indent=2))

In [ ]:
# Cell 13 — qualitative inspection
for i in range(min(2, len(eval_rows))):
    print('='*72)
    print('TRANSCRIPT:')
    print(eval_rows[i]['transcript'])
    print('\nPREDICTED:')
    print(preds[i])
    print('\nGOLD:')
    print(json.dumps(eval_rows[i]['soap'], indent=2))

In [ ]:
# Cell 14 — save adapter
ADAPTER_DIR = '/kaggle/working/adapter_v0'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('saved ->', ADAPTER_DIR)
!ls -la $ADAPTER_DIR

In [ ]:
# Cell 14b — plot training + held-out task curves; dump log_history.json
import json, matplotlib.pyplot as plt
from pathlib import Path

log = trainer.state.log_history
out_dir = Path('/kaggle/working')
(out_dir / 'log_history.json').write_text(json.dumps(log, indent=2))

train_e = [e for e in log if 'loss' in e and 'eval_loss' not in e]
eval_e  = [e for e in log if 'eval_loss' in e]

ts  = [e['step'] for e in train_e]
tl  = [e['loss'] for e in train_e]
gn  = [e.get('grad_norm') for e in train_e]
lr_ = [e.get('learning_rate') for e in train_e]
es  = [e['step'] for e in eval_e]
el  = [e['eval_loss'] for e in eval_e]

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ax[0,0].plot(ts, tl, label='train')
if el: ax[0,0].plot(es, el, 'o-', label='eval')
ax[0,0].set_title('loss'); ax[0,0].set_xlabel('step'); ax[0,0].legend()
ax[0,1].plot(ts, gn); ax[0,1].set_title('grad_norm'); ax[0,1].set_xlabel('step')
ax[1,0].plot(ts, lr_); ax[1,0].set_title('learning_rate'); ax[1,0].set_xlabel('step')
ax[1,1].axis('off')
fig.tight_layout(); fig.savefig(out_dir / 'curves_train.png', dpi=120); plt.show()

print('saved curves_train.png, log_history.json ->', out_dir)

## Export for local serving

Pipeline: LoRA adapter -> merge into base (fp16) -> convert HF -> GGUF (fp16) -> quantise -> Q4_K_M.

Output artefacts (download from `/kaggle/working/`):
- `qwen3b-soap-f16.gguf` (~6 GB) — full precision baseline
- `qwen3b-soap-q4_k_m.gguf` (~2 GB) — quantised, for local llama.cpp serving

In [ ]:
# Cell A — merge LoRA adapter into base, save full fp16 weights (CPU merge avoids VRAM spike)
import gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

# free GPU memory from the trained 4-bit model before loading fp16 base on CPU
del trainer, model
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cpu',
)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
MERGED_DIR = '/kaggle/working/merged_fp16'
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print('merged ->', MERGED_DIR)
!ls -la $MERGED_DIR

In [ ]:
# Cell B — clone llama.cpp and convert HF -> GGUF (fp16). Pip's red 'incompatible' lines
# are pre-existing Kaggle image dep conflicts, NOT errors from this conversion.
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/merged_fp16 \
    --outfile /kaggle/working/qwen3b-soap-f16.gguf \
    --outtype f16

In [ ]:
# Cell C — build llama-quantize and produce Q4_K_M
!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_CUDA=OFF 2>&1 | tail -5
!cd /kaggle/working/llama.cpp && cmake --build build --config Release -j --target llama-quantize 2>&1 | tail -5
!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    /kaggle/working/qwen3b-soap-f16.gguf \
    /kaggle/working/qwen3b-soap-q4_k_m.gguf \
    Q4_K_M
!ls -lh /kaggle/working/*.gguf

## v2 roadmap

Once the spike confirms the pipeline:

1. **Realistic transcripts** — 2000-2500 token messy dialogues (synthetic via GPT-4o, then hand-corrected sample).
2. **Multi-template training** — add referral letter spec; train on mixture; eval on held-out spec to prove template-awareness.
3. **Frozen eval set** — 50 examples (20 hand-curated + 30 synthetic-corrected), built before training.
4. **Stronger metrics** — per-field F1, schema validation via jsonschema, judge-model semantic scoring.
5. **Guardrails** — Outlines/jsonschema-constrained decoding, PII de-identification pre-step.
6. **DPO/RSFT** — once SFT plateaus.